In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import xarray as xr
import scipy.stats as stats
import matplotlib.pyplot as plt
import tqdm as tqdm
import sklearn 
import pickle

from scipy.integrate import simpson
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

from dask.distributed import Client, progress


In [ ]:
with open(r"/data/keeling/a/cstraus2/Phd_research/Apr_2_Oct_Station_events_data_multidf_negdeltas_del_normalized_priorpeaktogether2_delta_perPtot.pkl", "rb") as f:
   Apr_2_Oct_Station_events_dict = pickle.load(f)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

bbc_data = Apr_2_Oct_Station_events_dict['cmi']
Y = bbc_data.Peak_Precipitation_rate_in.values.reshape(-1,1)
X_prior = bbc_data.Prior_sm_10.values.reshape(-1,1)
X_delta = bbc_data.Delta_sm_10.values.reshape(-1,1)

model_prior_peak = LinearRegression()
model_prior_peak.fit(X_prior,Y)
model_delta_peak = LinearRegression()
model_delta_peak.fit(X_delta, Y)

prior_pred = model_prior_peak.predict(X_prior)
delta_pred = model_delta_peak.predict(X_delta)

r2_prior = r2_score(Y, prior_pred)
r2_delta = r2_score(Y, delta_pred)

print('r2 prior',r2_prior)
print('r2_delta', r2_delta)

fig, ax = plt.subplots(1,2, figsize = (10,6))
ax[0].scatter(bbc_data.Prior_sm_10, bbc_data.Peak_Precipitation_rate_in)
ax[0].plot(X_prior, prior_pred, color = 'red')
ax[0].set_title('Prior and Peak')
ax[0].set_xlabel('Prior sm 10')
ax[0].set_ylabel('Peak Precipitation')

ax[1].scatter( bbc_data.Delta_sm_10,bbc_data.Peak_Precipitation_rate_in)
ax[1].plot(X_delta,delta_pred, color = 'red')
ax[1].set_title('Delta and Peak')
ax[1].set_xlabel('Delta sm 10')
ax[1].set_ylabel('Peak Precipitation')


In [ ]:
def pdf_generation_trifirst(data,bw_scale):

    kde_combined_results =[]
    column_min_max_values = []
    band_widths_tri_vals = []
    
    #iterate through each station's data
    for s, df in data.items():
        #s is the station key in the dictionary and df is specific dataframe 
        #calculate individual kdes
        print('Station:', s)
        #df=df.dropna()

        #calculate trivariate kdes between deltas/ptot, peakP, prior sm 
        P_data = df.loc[:,df.columns.str.contains('Precipitation', regex = True)]
        prior_sm_data = df.loc[:,df.columns.str.contains('Prior', regex = True)]
        
        for i_P in range (P_data.shape[1]):
            P_data_i = P_data.iloc[:, i_P]
            P_col_label = P_data_i.name
            if 'Total' in P_col_label: #don't calc for delta/ptot with total P, so change delta cols pulled
                deltas_data = df.loc[:,df.columns.str.contains('Delta') & 
                                        ~df.columns.str.contains('Ptot')]
            else:
                deltas_data = df.loc[:,df.columns.str.contains('Delta', regex = True)]
            for i_sm in range(prior_sm_data.shape[1]):
                prior_sm_data_i = prior_sm_data.iloc[:,i_sm]
                prior_sm_col_label = prior_sm_data_i.name
                for j_Ptot in range(deltas_data.shape[1]):
                    deltas_Ptot_j = deltas_data.iloc[:, j_Ptot]
                    deltas_Ptot_col_label = deltas_Ptot_j.name
                    multi_data = np.vstack([
                        P_data_i.values, prior_sm_data_i.values, deltas_Ptot_j.values
                                        ])
                    tri_kde = stats.gaussian_kde(multi_data, bw_method = 'scott')
                    tri_kde.set_bandwidth(bw_method = tri_kde.scotts_factor() * bw_scale)
                    tri_kde_bw_x = tri_kde.factor * np.sqrt(tri_kde.covariance[0,0])
                    tri_kde_bw_y = tri_kde.factor * np.sqrt(tri_kde.covariance[1,1])
                    tri_kde_bw_z = tri_kde.factor * np.sqrt(tri_kde.covariance[2,2])
                    tri_bw = [tri_kde_bw_x, tri_kde_bw_y, tri_kde_bw_z]
                    trivar_label = f'trivar_kde_{P_col_label}_{prior_sm_col_label}_{deltas_Ptot_col_label}'
                    kde_combined_results.append((s, trivar_label, tri_kde, 'trivariate'))
                    band_widths_tri_vals.append((s, tri_bw, trivar_label))


        #iterate through each column in the station's data
        for col in df.columns:
            #print('column', df[col])
            #print('nan?',df[col].isna().sum())
            #print('inf?',np.isinf(df[col]).sum())
            col_min = np.round(df[col].min(), decimals=2)
            col_max = np.round(df[col].max(), decimals=2)
            column_min_max_values.append((s, col, col_min, col_max))

            
    #check shape of combined dataframe
    #for i, item in enumerate(kde_combined_results):
    #    print(f"Item {i}: length = {len(item)}, content = {item}")

    #convert results into a multiIndex df
    multi_kde_df = pd.DataFrame(kde_combined_results, columns = ['Station', 'Variable', 'KDE_Object', 
                                                                 'Type'])
    multi_kde_df.set_index(['Station', 'Variable', 'Type'], inplace = True)

    min_max_df = pd.DataFrame(column_min_max_values, columns=['Station', 'Variable', 'Min', 'Max'])
    min_max_df.set_index(['Station', 'Variable'], inplace=True)
    
    band_widths_tri_df = pd.DataFrame(band_widths_tri_vals, columns = ['Station', 'bandwidth', 
                                                                       'Variable'])
    
    return multi_kde_df, min_max_df, band_widths_tri_df


In [ ]:
def calc_entropy_trifirst(kde_df_Series_data, min_max_df,var_bw):
    entropies = {}
    kde_df_Series_data = kde_df_Series_data.sort_index() 
    resolution_num = 31 #must be an odd number so there are even number of intervals for the simpsons rule integration <- normalization has been removed
    n_bw = 15
    
    for s, group in kde_df_Series_data.groupby(level = 'Station'):
        print(s)

        station_kde_Series = kde_df_Series_data.loc[(s, 'KDE_Object')]
        n_rows = len(station_kde_Series)
        station_entropy_values = {}
        
        station_cols_min_vals = min_max_df.Min.loc[s]
        station_cols_max_vals = min_max_df.Max.loc[s]
        keep_vars = station_cols_min_vals.index.str.contains('Precipitation|Prior|Delta', regex = True)
        filtered_min_vals = station_cols_min_vals[keep_vars]
        filtered_max_vals = station_cols_max_vals[keep_vars]

        #print(f"Filtered min values: {filtered_min_vals}")
        #print(f"Filtered max values: {filtered_max_vals}")

        if filtered_min_vals.empty or filtered_max_vals.empty:
            print(f"⚠️ No valid min/max values for {s}")
            continue
               
        for (tri_label, trivar_kde_obj) in station_kde_Series.items():
            #print(f"Processing tri KDE for {tri_label}")
            tri_kde_name = tri_label[0].replace("trivar_kde_", "")
            delta_var_label = tri_kde_name[tri_kde_name.find('Delta'):]
            prior_var_label = tri_kde_name[tri_kde_name.find('Prior'):tri_kde_name.find('_Delta')]
            P_var_label = tri_kde_name[:tri_kde_name.find('_Prior')]
            try:
                x_min = filtered_min_vals[P_var_label]
                x_max = filtered_max_vals[P_var_label]
                y_min = filtered_min_vals[prior_var_label]
                y_max = filtered_max_vals[prior_var_label]
                z_min = filtered_min_vals[delta_var_label]
                z_max = filtered_max_vals[delta_var_label]

            except KeyError:
                print(f"⚠️ Could not find min/max for some variable")
                continue
                    
            var_bw_row = var_bw.loc[(var_bw['Station'] == s) & 
                                              (var_bw['Variable'] == tri_label[0])]
                    
            x_tri_bw, y_tri_bw, z_tri_bw = var_bw_row['bandwidth'].values[0]
                    
            x_vals_P = np.linspace(x_min - x_tri_bw*n_bw, x_max + x_tri_bw*n_bw, resolution_num) 
            dx = x_vals_P[1]-x_vals_P[0]
                    
            y_vals_prior = np.linspace(y_min - y_tri_bw*n_bw, y_max + y_tri_bw*n_bw, resolution_num)
            dy = y_vals_prior[1]-y_vals_prior[0]

            z_vals_delta = np.linspace(z_min - z_tri_bw*n_bw, z_max + z_tri_bw*n_bw, resolution_num)
            dz = z_vals_delta[1]-z_vals_delta[0]

            #pair data in meshgrd and flatten to 3xN array for kde
            X_P, Y_priorsm, Z_deltaPtot =np.meshgrid(x_vals_P, y_vals_prior,z_vals_delta, 
                                                     indexing = 'ij')
            positions = np.vstack([X_P.ravel(), Y_priorsm.ravel(), Z_deltaPtot.ravel()])
            densities = np.reshape(trivar_kde_obj(positions), X_P.shape) #(x,y,z) = (P, prior, delta)
            
            #print('pre-normalized D3_pdf', np.sum(densities * dx * dy * dz))


            # Normalize the density (integrate over all grid points (triple integral))
            #triple_integral = simpson(simpson(simpson(densities, dx=dz, axis = 2), dx = dy, axis = 1), dx=dx, axis = 0) #should sum to 1
            D3_pdf = densities #/ triple_integral  # Normalize to get probability distribution
            #print('normalized D3_pdf', np.sum(D3_pdf * dx * dy * dz))

            edge_vals = np.concatenate([
                    D3_pdf[0, :, :].ravel(), D3_pdf[-1, :, :].ravel(),  # Top and bottom edges
                    D3_pdf[:, 0, :].ravel(), D3_pdf[:, -1, :].ravel(),  # Left and right edges
                    D3_pdf[:, :, 0].ravel(), D3_pdf[:, :, -1].ravel()]) # Front and back edges
            #max_edge_density = np.max(edge_vals)
            #print(f"🔍 Max density on edges of tri PDF between {P_var_label} and {prior_var_label} and {delta_var_label}: {max_edge_density:.2e}")
            #if max_edge_density > 1e-5:
                #print("⚠️ Significant density near the edge — consider extending x/y/z " \
                #"ranges further,  ideally edge range below 1e-5")

           # print('dx,dy,dz', (dx,dy,dz))
            #print('bin num for tri', resolution_num)

            # Compute joint probabilities using dx * dy integration
            D3_pvals = D3_pdf * dx * dy * dz  # Discretized probability mass function
            #print('total mass = ', np.sum(D3_pvals)) #should be very close to 1

            # Compute log2 probabilities, avoiding log(0) by masking zeros
            D3_pvals_flat = D3_pvals.ravel()
            #if np.any(D3_pvals_flat <= 0):
            #    print("Warning: PDF has near-zero or negative values")

            #add to small probabilities to stabilize the log of the small values in tails
            eps = 1e-12  
            log2_pvals = np.log2(D3_pvals_flat + eps)

            # Calculate trivar entropy
            H_tri = np.round(-np.sum(D3_pvals_flat * log2_pvals), decimals = 3)
            #if np.isnan(H_tri) or np.isinf(H_tri):
            #    print("Entropy result is unstable")

            #print(f'trivariate entropy between {P_var_label} and {prior_var_label} and {delta_var_label}: {H_tri}')
            station_entropy_values[f'trivariate_entropy_{P_var_label}_{prior_var_label}_{delta_var_label}'] = H_tri

            #calculate the joint pdf by summing along the joint axes
            #(x,y,z) = (P, prior, delta)
            #the following are densities
            joint_P_prior_pdf = np.sum(D3_pdf, axis = 2) * dz
            #print('Check ∫pXY =', np.sum(joint_P_prior_pdf * dx * dy))
            joint_prior_delta_pdf = np.sum(D3_pdf, axis = 0) * dx
            #print('Check ∫pXY =', np.sum(joint_prior_delta_pdf * dy * dz))
            joint_P_delta_pdf = np.sum(D3_pdf, axis = 1) * dy
            #print('Check ∫pXY =', np.sum(joint_P_delta_pdf * dx * dz))

            #joint_P_prior_pdf /= np.sum(joint_P_prior_pdf * dx * dy)
            #joint_prior_delta_pdf /= np.sum(joint_prior_delta_pdf * dy * dz)
            #joint_P_delta_pdf /= np.sum(joint_P_delta_pdf * dx * dz)

            #calculate the joint entropies
            #calc the probabilities
            joint_P_prior_pvals = joint_P_prior_pdf * dx * dy
            joint_prior_delta_pvals = joint_prior_delta_pdf * dy * dz
            joint_P_delta_pvals = joint_P_delta_pdf * dx * dz

            joint_P_prior_pvals = joint_P_prior_pvals.ravel()
            joint_prior_delta_pvals = joint_prior_delta_pvals.ravel()
            joint_P_delta_pvals = joint_P_delta_pvals.ravel()

            H_P_prior = -np.sum(joint_P_prior_pvals * np.log2(joint_P_prior_pvals + eps))
            H_prior_delta = -np.sum(joint_prior_delta_pvals * np.log2(joint_prior_delta_pvals + eps))
            H_P_delta = -np.sum(joint_P_delta_pvals * np.log2(joint_P_delta_pvals + eps))

            station_entropy_values[f'joint_entropy_{P_var_label}_{prior_var_label}'] = np.round(H_P_prior, 3)
            station_entropy_values[f'joint_entropy_{prior_var_label}_{delta_var_label}'] = np.round(H_prior_delta, 3)
            station_entropy_values[f'joint_entropy_{P_var_label}_{delta_var_label}'] = np.round(H_P_delta, 3)

            #individual pdfs
            P_pdf = np.sum(np.sum(D3_pdf, axis = 2), axis =1) * dy * dz
            #print('Check ∫pX =', np.sum(P_pdf * dx ))
            prior_pdf = np.sum(np.sum(D3_pdf, axis = 2), axis =0) * dx * dz
            #print('Check ∫pY =', np.sum(prior_pdf * dy ))
            delta_pdf = np.sum(np.sum(D3_pdf, axis = 1), axis =0) * dx * dy
            #print('Check ∫pZ =', np.sum(delta_pdf * dz ))

            #P_pdf /= np.sum(P_pdf * dx)
            #prior_pdf /= np.sum(prior_pdf * dy)
            #delta_pdf /= np.sum(delta_pdf * dz)

            #calculate ind probabilites
            P_pvals = P_pdf * dx
            prior_pvals = prior_pdf * dy
            delta_pvals = delta_pdf * dz

            H_P = -np.sum(P_pvals * np.log2(P_pvals + eps))
            H_prior = -np.sum(prior_pvals * np.log2(prior_pvals + eps))
            H_delta = -np.sum(delta_pvals * np.log2(delta_pvals + eps))

            station_entropy_values[f'individual_entropy_{P_var_label}'] = np.round(H_P, 3)
            station_entropy_values[f'individual_entropy_{prior_var_label}'] = np.round(H_prior, 3)
            station_entropy_values[f'individual_entropy_{delta_var_label}'] = np.round(H_delta, 3)

        if station_entropy_values:
            entropy_df = pd.DataFrame([station_entropy_values])  # one row
            entropies[s] = entropy_df
        else:
            print(f"⚠️ No entropy values computed for station {s}")
    return entropies

In [ ]:
def mutual_information_calc(entropies_dict):
    Stations = list(entropies_dict.keys())
    Station_MI = []
    Station_tri_MI =[]

    for s in Stations:
        #print('station', s)
        station_entropy_df = entropies_dict[s]
        # Step 1: Get columns for H(X) and H(Y) (without 'joint_' prefix)
        individual_columns = [col for col in station_entropy_df.columns if ('individual_')  in col]

        # Step 2: Get columns for H(X, Y) (with 'joint_' prefix)
        joint_columns = [col for col in station_entropy_df.columns if 'joint_' in col]

        # Step 3: Get columns for H(X,Y,Z) (with 'trivariate' prefix)
        trivar_columns = [col for col in station_entropy_df.columns if 'trivariate_' in col]

        for joint_col in joint_columns:
            found = False
            joint_label = joint_col.replace('joint_entropy_', '')
            #print('Joint col', joint_col)
            #print('Joint label', joint_label)

            for col_x in individual_columns:
                for col_y in individual_columns:
                    if col_x == col_y:
                        continue  # skip self-pairs

                    # Remove the '_entropy' suffix to get base names
                    col_x_base = col_x.removeprefix('individual_entropy_')
                    col_y_base = col_y.removeprefix('individual_entropy_')

                    expected_joint_col = f"joint_entropy_{col_x_base}_{col_y_base}"

                    if expected_joint_col == joint_col:
                        #print('match found')
                        # Find the corresponding individual entropy columns
                        # Match found — calculate MI
                        H_X = station_entropy_df[col_x].iloc[0]
                        #print('H_X', H_X)
                        H_Y = station_entropy_df[col_y].iloc[0]
                        #print('H_Y', H_Y)
                        H_XY = station_entropy_df[joint_col].iloc[0]
                        #print('H_XY', H_XY)

                        MI = H_X + H_Y - H_XY
                        MI_label = 'MI_'+ col_x_base + '_and_' + col_y_base

                        Cxy = np.round(MI/H_Y, decimals = 3)
                        Cyx = np.round(MI/H_X, decimals = 3)

                        Station_MI.append({
                                "station": s,
                                "MI_label": MI_label,
                                "mutual_information": MI,
                                "H(x)": col_x_base,
                                "C_yx": Cyx,
                                "H(y)": col_y_base,
                                "C_xy": Cxy
                                        })

                        found = True
                        break  # inner loop
                if found:
                    break  # outer loop   
        #print('finished joint and individuals')
        #calculate for trivariate
        for trivar in trivar_columns:
            x_var = trivar[len('trivariate_entropy_'):trivar.find('_Prior')]
            y_var = trivar[trivar.find('Prior'):trivar.find('_Delta')]
            z_var = trivar[trivar.find('Delta'):]

            variable_labels = [x_var, y_var, z_var]
            individual_entropies_labels= ['individual_entropy_' + var for var in variable_labels]
            joint_combintations = [x_var + '_' + y_var, x_var + '_' + z_var, y_var + '_' + z_var]
            joint_entropies_labels = ['joint_entropy_' + var for var in joint_combintations]

            trivariate_entropy_label = trivar

            #H(x,y,z) + H(x) +H (y) + H(z) - H(x,y) - H(x,z) - H(y,z)
            Hx, Hy, Hz = [station_entropy_df[col].iloc[0] for col in individual_entropies_labels]
            Hxy, Hyz, Hxz = [station_entropy_df[col].iloc[0] for col in joint_entropies_labels]
            Hxyz = station_entropy_df[trivariate_entropy_label].iloc[0]

            MI_label = 'trivariate_MI_'+ x_var + '_'+ y_var + '_' + z_var
            Interacting_I = Hxyz + Hx + Hy + Hz - Hxy - Hyz - Hxz
            #Dependency_corr_I_label   = 'trivariate_DependencyCorr_I_'+ x_var + '_'+ y_var + '_' + z_var
            Dependency_I = Hx + Hy + Hz - Hxyz
            #Conditional_I_label = 'trivariate_Conditional_I_' + z_var + 'given' + x_var + '_'+ y_var
            Conditional_I = Hz + Hxy - Hxyz
            
            #print(Hxyz)
            C_tx = np.round(Conditional_I/Hx, decimals = 3)
            C_ty = np.round(Conditional_I/Hy, decimals = 3)
            C_tz = np.round(Conditional_I/Hz, decimals = 3)
            
            Station_tri_MI.append({
                                "station": s,
                                "MI_label": MI_label,
                                "Interacting_information": Interacting_I,
                                "Total_shared_information": Dependency_I,
                                "Z_Conditional_MI": Conditional_I,
                                "H(x)": individual_entropies_labels[0],
                                "C_tx": C_tx,
                                "H(y)": individual_entropies_labels[1] ,
                                "C_ty": C_ty,
                                "H(z)": individual_entropies_labels[2],
                                "C_tz": C_tz,
                                })



    return pd.DataFrame(Station_MI), pd.DataFrame(Station_tri_MI)

### Shuffle one of the two variables: Peak Precipitation rate or Prior SM 10/100 to break dependency in joint kde and maintain marginal sizing run code and evaluate for change from shuffling (MI should be reduced if calculations are correct and the dependency is not faulty)

In [ ]:
#shuffling code to break dependencies
if 'client' not in globals():
    client = Client()  # Replace with Client("<scheduler-address>:8786") to use an existing Dask cluster.

stations = list(Apr_2_Oct_Station_events_dict.keys())
n_iter = 500
mi_peak_sm10_df  = pd.DataFrame(index=stations, columns=range(n_iter), dtype=float)
mi_peak_sm100_df = pd.DataFrame(index=stations, columns=range(n_iter), dtype=float)
mi_tot_sm10_df  = pd.DataFrame(index=stations, columns=range(n_iter), dtype=float)
mi_tot_sm100_df = pd.DataFrame(index=stations, columns=range(n_iter), dtype=float)
mi_prior10_sm10_df  = pd.DataFrame(index=stations, columns=range(n_iter), dtype=float)
mi_prior10_sm100_df = pd.DataFrame(index=stations, columns=range(n_iter), dtype=float)
mi_prior100_sm10_df  = pd.DataFrame(index=stations, columns=range(n_iter), dtype=float)
mi_prior100_sm100_df = pd.DataFrame(index=stations, columns=range(n_iter), dtype=float)

mi_peak_sm10perPtot_df = pd.DataFrame(index=stations, columns=range(n_iter), dtype=float)
mi_peak_sm100perPtot_df = pd.DataFrame(index=stations, columns=range(n_iter), dtype=float)
mi_prior10_sm10perPtot_df = pd.DataFrame(index=stations, columns=range(n_iter), dtype=float)
mi_prior10_sm100perPtot_df = pd.DataFrame(index=stations, columns=range(n_iter), dtype=float)
mi_prior100_sm10perPtot_df = pd.DataFrame(index=stations, columns=range(n_iter), dtype=float)
mi_prior100_sm100perPtot_df = pd.DataFrame(index=stations, columns=range(n_iter), dtype=float)

mi_frames = {
    'MI_Peak_Precipitation_rate_in_and_Delta_sm_10': mi_peak_sm10_df,
    'MI_Peak_Precipitation_rate_in_and_Delta_sm_100': mi_peak_sm100_df,
    'MI_Total_Precipitation_in_and_Delta_sm_10': mi_tot_sm10_df,
    'MI_Total_Precipitation_in_and_Delta_sm_100': mi_tot_sm100_df,
    'MI_Prior_sm_10_and_Delta_sm_10': mi_prior10_sm10_df,
    'MI_Prior_sm_10_and_Delta_sm_100': mi_prior10_sm100_df,
    'MI_Prior_sm_100_and_Delta_sm_10': mi_prior100_sm10_df,
    'MI_Prior_sm_100_and_Delta_sm_100': mi_prior100_sm100_df,
    'MI_Peak_Precipitation_rate_in_and_Delta_sm_10_perPtot': mi_peak_sm10perPtot_df,
    'MI_Peak_Precipitation_rate_in_and_Delta_sm_100_perPtot': mi_peak_sm100perPtot_df,
    'MI_Prior_sm_10_and_Delta_sm_10_perPtot': mi_prior10_sm10perPtot_df,
    'MI_Prior_sm_10_and_Delta_sm_100_perPtot': mi_prior10_sm100perPtot_df,
    'MI_Prior_sm_100_and_Delta_sm_10_perPtot': mi_prior100_sm10perPtot_df,
    'MI_Prior_sm_100_and_Delta_sm_100_perPtot': mi_prior100_sm100perPtot_df,
}

def run_shuffle_iteration(iter_idx):
    rng = np.random.default_rng(iter_idx)
    shuffled_data = {}
    for s, df in Apr_2_Oct_Station_events_dict.items():
        station_shuffled_data = df.copy()
        station_shuffled_data['Delta_sm_10'] = rng.permutation(df['Delta_sm_10'].values)
        station_shuffled_data['Delta_sm_100'] = rng.permutation(df['Delta_sm_100'].values)
        station_shuffled_data['Delta_sm_10_perPtot'] = rng.permutation(df['Delta_sm_10_perPtot'].values)
        station_shuffled_data['Delta_sm_100_perPtot'] = rng.permutation(df['Delta_sm_100_perPtot'].values)
        shuffled_data[s] = station_shuffled_data

    Apr_2_Oct_multi_station_kdes_shuffled, Apr_2_Oct_multi_station_min_max_df, Apr_2_Oct_multi_station_trivar_bw = pdf_generation_trifirst(shuffled_data, bw_scale = 1)
    shuffled_Entropies = calc_entropy_trifirst(
        Apr_2_Oct_multi_station_kdes_shuffled,
        Apr_2_Oct_multi_station_min_max_df,
        Apr_2_Oct_multi_station_trivar_bw
    )
    MIs_shuffled, tri_MIs_shuffled = mutual_information_calc(shuffled_Entropies)
    mi_series = MIs_shuffled.set_index(['station', 'MI_label'])['mutual_information']
    return iter_idx, mi_series

futures = client.map(run_shuffle_iteration, range(n_iter))
progress(futures)
mi_results = client.gather(futures)

for iter_idx, mi_series in mi_results:
    for s in stations:
        for label, df in mi_frames.items():
            df.loc[s, iter_idx] = mi_series.loc[(s, label)]
